# KazNLP NER Demo

This notebook demonstrates how to use the KazBERT + BiLSTM + CRF model for Named Entity Recognition.

In [ ]:
from transformers import AutoTokenizer
from kaznlp.models.ner_model import KazBERT_BiLSTM_CRF_NER
import torch


In [ ]:
tag2idx = {"O": 0, "B-LOC": 1, "I-LOC": 2, "B-PER": 3, "I-PER": 4, "B-ORG": 5, "I-ORG": 6}
idx2tag = {v: k for k, v in tag2idx.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("ai-forever/mbert-base-kaznlp")
model = KazBERT_BiLSTM_CRF_NER("ai-forever/mbert-base-kaznlp", tag2idx)
model.load_state_dict(torch.load("kazner_model.pt", map_location=device))
model.to(device)
model.eval()

In [ ]:
text = "Астана қаласы Қазақстанның астанасы болып табылады"
tokens = text.split()
inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", padding="max_length", truncation=True, max_length=128)

with torch.no_grad():
    predictions = model(inputs["input_ids"].to(device), attention_mask=inputs["attention_mask"].to(device))

print([(t, idx2tag[p]) for t, p in zip(tokens, predictions[0][:len(tokens)])])